# Week 51, Day 4: Minimum spanning trees

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w051d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
edges = [(4, "A", "B"), (1, "A", "C"), (3, "B", "C"), (2, "B", "D"), (5, "C", "D")]

group = {v: v for v in "ABCD"}
kept = []
for w, u, v in sorted(edges):
    if group[u] != group[v]:
        old, new = group[v], group[u]
        for x in group:
            if group[x] == old:
                group[x] = new
        kept.append((u, v, w))
print(kept, sum(w for _, _, w in kept))

In [ ]:
def kruskal(vertices, edges):
    parent = {v: v for v in vertices}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    tree, total = [], 0
    for w, u, v in sorted(edges):
        ru, rv = find(u), find(v)
        if ru != rv:
            parent[ru] = rv
            tree.append((u, v, w))
            total += w
            if len(tree) == len(parent) - 1:
                break
    return total, tree

villages = "ABCDEF"
roads = [(7, "A", "B"), (5, "A", "D"), (8, "B", "C"), (9, "B", "D"), (7, "B", "E"),
         (5, "C", "E"), (15, "D", "E"), (6, "D", "F"), (8, "E", "F"), (11, "F", "C")]
print(kruskal(villages, roads))

In [ ]:
import heapq

def prim(graph, start):
    in_tree, tree, total = {start}, [], 0
    heap = [(w, start, v) for v, w in graph[start].items()]
    heapq.heapify(heap)
    while heap and len(in_tree) < len(graph):
        w, u, v = heapq.heappop(heap)
        if v in in_tree:
            continue                     # this edge now joins two tree vertices
        in_tree.add(v)
        tree.append((u, v, w))
        total += w
        for x, wx in graph[v].items():
            if x not in in_tree:
                heapq.heappush(heap, (wx, v, x))
    return total, tree

graph = {}
for w, u, v in [(7, "A", "B"), (5, "A", "D"), (8, "B", "C"), (9, "B", "D"), (7, "B", "E"),
                (5, "C", "E"), (15, "D", "E"), (6, "D", "F"), (8, "E", "F"), (11, "F", "C")]:
    graph.setdefault(u, {})[v] = w
    graph.setdefault(v, {})[u] = w
print(prim(graph, "A"))

In [ ]:
def clusters(points, k):
    def dist(a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])
    names = list(points)
    edges = sorted((dist(points[a], points[b]), a, b) for i, a in enumerate(names) for b in names[i + 1:])
    parent = {n: n for n in names}
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    groups = len(names)
    for d, a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            continue
        if groups == k:
            spacing = d                          # the closest pair in different clusters
            break
        parent[ra] = rb
        groups -= 1
    out = {}
    for n in names:
        out.setdefault(find(n), []).append(n)
    return sorted(sorted(g) for g in out.values()), spacing

shops = {"a": (0, 0), "b": (1, 0), "c": (0, 2), "d": (9, 9), "e": (10, 8), "f": (20, 0)}
print(clusters(shops, 3))

> **Key idea.** A spanning tree connects every vertex with V − 1 edges and no cycle; a minimum spanning tree has the least total weight (unique if all weights differ). Kruskal's algorithm sorts edges and keeps each one whose ends are in different union-find components: O(E log E). Prim's algorithm grows one tree, always adding the cheapest edge leaving it, with a heap: O(E log V); for dense graphs, an O(V²) scan without a heap is better. Both are correct because of the cut property: the cheapest edge across any cut is in some MST. MSTs give the cheapest network, maximum spanning trees the strongest, MST paths the best bottlenecks, and stopping Kruskal's algorithm at k components gives single-linkage clustering.

### Exercise: Kruskal's algorithm

Write `kruskal(vertices, edges)` for edges given as `(u, v, weight)`: a pair `(total, tree)` with the tree's edges in the order they were kept, using union-find (path halving and union by size). Ties in weight keep the edges' original order (use a stable sort by weight). Raise `ValueError` if the graph isn't connected. Then `forest(vertices, edges)`: the minimum spanning forest's total and its number of trees.

In [ ]:
def kruskal(vertices, edges):
    ___

def forest(vertices, edges):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef brute(vs, es):\n    best = None\n    n = len(vs)\n    for combo in itertools.combinations(es, n - 1):\n        parent = {v: v for v in vs}\n        def find(x):\n            while parent[x] != x:\n                x = parent[x]\n            return x\n        ok = True\n        for u, v, w in combo:\n            if find(u) == find(v):\n                ok = False\n                break\n            parent[find(u)] = find(v)\n        if ok:\n            t = sum(w for _, _, w in combo)\n            best = t if best is None else min(best, t)\n    return best\ndef test_kruskal():\n    \"trees\"\n    roads = [(\"A\", \"B\", 7), (\"A\", \"D\", 5), (\"B\", \"C\", 8), (\"B\", \"D\", 9), (\"B\", \"E\", 7), (\"C\", \"E\", 5), (\"D\", \"E\", 15), (\"D\", \"F\", 6), (\"E\", \"F\", 8), (\"F\", \"C\", 11)]\n    total, tree = kruskal(\"ABCDEF\", roads)\n    assert total == 30 and tree == [(\"A\", \"D\", 5), (\"C\", \"E\", 5), (\"D\", \"F\", 6), (\"A\", \"B\", 7), (\"B\", \"E\", 7)]\n    try:\n        kruskal(\"ABC\", [(\"A\", \"B\", 1)])\n        assert False\n    except ValueError:\n        pass\n    assert forest(\"ABCD\", [(\"A\", \"B\", 1), (\"C\", \"D\", 2), (\"A\", \"B\", 0)]) == (2, 2) and kruskal(\"A\", []) == (0, [])\n    rng = random.Random(1)\n    for _ in range(60):\n        vs = list(range(rng.randint(2, 5)))\n        es = [(a, b, rng.randint(1, 9)) for a in vs for b in vs if a < b and rng.random() < 0.7]\n        want = brute(vs, es)\n        if want is None:\n            continue\n        total, tree = kruskal(vs, es)\n        assert total == want and len(tree) == len(vs) - 1")

### Exercise: Prim's algorithm

Write `prim(graph, start)` for a dict-of-dicts undirected graph: `(total, tree)`, the tree's edges `(u, v, weight)` in the order added, using a heap of `(weight, u, v)` and skipping edges whose far end is already in the tree; raise `ValueError` if not every vertex is reached. Then `same_total(graph)`: whether Prim's algorithm from every start vertex gives the same total (it always should).

In [ ]:
import heapq

def prim(graph, start):
    ___

def same_total(graph):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_prim():\n    \"prim\"\n    graph = {}\n    for u, v, w in [(\"A\", \"B\", 7), (\"A\", \"D\", 5), (\"B\", \"C\", 8), (\"B\", \"D\", 9), (\"B\", \"E\", 7), (\"C\", \"E\", 5), (\"D\", \"E\", 15), (\"D\", \"F\", 6), (\"E\", \"F\", 8), (\"F\", \"C\", 11)]:\n        graph.setdefault(u, {})[v] = w\n        graph.setdefault(v, {})[u] = w\n    total, tree = prim(graph, \"A\")\n    assert total == 30 and tree[:3] == [(\"A\", \"D\", 5), (\"D\", \"F\", 6), (\"A\", \"B\", 7)] and len(tree) == 5\n    assert same_total(graph)\n    try:\n        prim({\"A\": {\"B\": 1}, \"B\": {\"A\": 1}, \"C\": {}}, \"A\")\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(2)\n    for _ in range(50):\n        n = rng.randint(2, 8)\n        g = {v: {} for v in range(n)}\n        for v in range(1, n):\n            u = rng.randrange(v)\n            g[u][v] = g[v][u] = rng.randint(1, 9)\n        for _ in range(n):\n            a, b = rng.sample(range(n), 2)\n            g[a][b] = g[b][a] = rng.randint(1, 9)\n        assert same_total(g)")

### Exercise: Connecting points

Points on a map can each be joined to any other at a cost equal to their **Manhattan distance** (|x1 − x2| + |y1 − y2|). Write `connect_points(points)`: the least total cost to connect them all, with the O(V²) version of Prim's algorithm (a dict of each outside point's cheapest link to the tree, scanned for the minimum each step; never build the list of all pairs). Then `longest_link(points)`: the longest single link in that minimum network. The checks use 1,500 points.

In [ ]:
def connect_points(points):
    ___

def longest_link(points):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_points():\n    \"points\"\n    pts = [(0, 0), (2, 2), (3, 10), (5, 2), (7, 0)]\n    assert connect_points(pts) == 20 and longest_link(pts) == 9\n    assert connect_points([(1, 1)]) == 0 and connect_points([]) == 0\n    rng = random.Random(3)\n    big = [(rng.randint(0, 10**6), rng.randint(0, 10**6)) for _ in range(1500)]\n    t0 = time.perf_counter()\n    connect_points(big)\n    assert time.perf_counter() - t0 < 8")

### Exercise: Clustering

Write `clusters(points, k)` for points in a dict from name to `(x, y)`: the k single-linkage clusters by Manhattan distance (each a sorted list, the list sorted), and the **spacing**: the smallest distance between two points in different clusters. Stop Kruskal's algorithm when k groups remain. Then `best_k(points, min_spacing)`: the largest k for which the spacing is at least `min_spacing` (1 if even 2 clusters fall short).

In [ ]:
def clusters(points, k):
    ___

def best_k(points, min_spacing):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_clusters():\n    \"clusters\"\n    shops = {\"a\": (0, 0), \"b\": (1, 0), \"c\": (0, 2), \"d\": (9, 9), \"e\": (10, 8), \"f\": (20, 0)}\n    groups, spacing = clusters(shops, 3)\n    assert groups == [[\"a\", \"b\", \"c\"], [\"d\", \"e\"], [\"f\"]] and spacing == 16\n    assert clusters(shops, 2) == ([[\"a\", \"b\", \"c\", \"d\", \"e\"], [\"f\"]], 18)\n    assert clusters(shops, 6)[1] == 1\n    assert best_k(shops, 16) == 3 and best_k(shops, 17) == 2 and best_k(shops, 100) == 1 and best_k(shops, 1) == 6")

### Exercise: The strongest links

Links between servers have capacities. Write `max_spanning_tree(vertices, links)`: `(total, tree)` for the spanning tree with the **largest** total capacity (Kruskal's algorithm, sorting by capacity, largest first). Then `best_bottleneck(vertices, links, a, b)`: the largest capacity c such that a and b can be joined using only links of capacity at least c, read off the maximum spanning tree (the smallest capacity on the tree path from a to b).

In [ ]:
def max_spanning_tree(vertices, links):
    ___

def best_bottleneck(vertices, links, a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef test_max():\n    \"maximum\"\n    links = [(\"A\", \"B\", 10), (\"A\", \"C\", 3), (\"B\", \"C\", 6), (\"C\", \"D\", 8), (\"B\", \"D\", 2)]\n    total, tree = max_spanning_tree(\"ABCD\", links)\n    assert total == 24 and sorted(tree) == [(\"A\", \"B\", 10), (\"B\", \"C\", 6), (\"C\", \"D\", 8)]\n    assert best_bottleneck(\"ABCD\", links, \"A\", \"D\") == 6 and best_bottleneck(\"ABCD\", links, \"C\", \"D\") == 8\n    rng = random.Random(4)\n    for _ in range(60):\n        vs = list(range(5))\n        ls = [(a, b, rng.randint(1, 20)) for a in vs for b in vs if a < b and rng.random() < 0.6]\n        parent = {v: v for v in vs}\n        def find(x):\n            while parent[x] != x:\n                x = parent[x]\n            return x\n        for u, v, w in ls:\n            parent[find(u)] = find(v)\n        if len({find(v) for v in vs}) > 1:\n            continue\n        a, b = rng.sample(vs, 2)\n        best = 0\n        for c in sorted({w for _, _, w in ls}):\n            parent2 = {v: v for v in vs}\n            def f2(x):\n                while parent2[x] != x:\n                    x = parent2[x]\n                return x\n            for u, v, w in ls:\n                if w >= c:\n                    parent2[f2(u)] = f2(v)\n            if f2(a) == f2(b):\n                best = c\n        assert best_bottleneck(vs, ls, a, b) == best")

### Bug hunt: Keeping every cheap edge

This "Kruskal" keeps edges cheapest first but never checks whether an edge closes a cycle, so it simply keeps the three cheapest: A-B, A-C and B-C, which form a loop and leave D out. It prints `5`, the total of something that isn't a spanning tree. Check components with union-find so it prints `7`.

In [ ]:
edges = [("A", "B", 1), ("A", "C", 2), ("B", "C", 2), ("C", "D", 4), ("B", "D", 5)]
kept = []
for u, v, w in sorted(edges, key=lambda e: e[2]):
    kept.append((u, v, w))
    if len(kept) == 3:
        break
print(sum(w for _, _, w in kept))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no cycles\"\n    expect_output(\"7\")")

### Bug hunt: Heaviest first

This Kruskal sorts the edges with `reverse=True`, so it builds the **maximum** spanning tree: this prints `11`. Sort cheapest first so it prints `7`.

In [ ]:
edges = [("A", "B", 1), ("A", "C", 2), ("B", "C", 2), ("C", "D", 4), ("B", "D", 5)]
parent = {v: v for v in "ABCD"}

def find(x):
    while parent[x] != x:
        x = parent[x]
    return x

total = 0
for u, v, w in sorted(edges, key=lambda e: e[2], reverse=True):
    if find(u) != find(v):
        parent[find(u)] = find(v)
        total += w
print(total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"cheapest first\"\n    expect_output(\"7\")")

### Bug hunt: Joining the tree to itself

This Prim's algorithm doesn't skip edges whose far end is already in the tree, so an edge between two tree vertices gets added and its weight counted: this prints `9`. Skip those edges so it prints `7`.

In [ ]:
import heapq

graph = {"A": {"B": 1, "C": 2}, "B": {"A": 1, "C": 2, "D": 5}, "C": {"A": 2, "B": 2, "D": 4}, "D": {"B": 5, "C": 4}}
in_tree, total = {"A"}, 0
heap = [(w, "A", v) for v, w in graph["A"].items()]
heapq.heapify(heap)
added = 0
while heap and added < len(graph) - 1:
    w, u, v = heapq.heappop(heap)
    in_tree.add(v)
    total += w
    added += 1
    for x, wx in graph[v].items():
        if x not in in_tree:
            heapq.heappush(heap, (wx, v, x))
print(total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"skip edges inside the tree\"\n    expect_output(\"7\")")

### Puzzle: Kruskal's algorithm

Arrange the lines into Kruskal's algorithm (a `find` with path halving is given). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Undirected edges: A-B 3, A-C 1, B-C 2, B-D 4, C-D 5, D-E 1.

p = ___   # the number of edges in any spanning tree of these 5 vertices
q = ___   # the total weight of the MST
r = ___   # the edges Kruskal's algorithm skips, as a sorted list of strings like "A-B"
s = ___   # the first edge Prim's algorithm adds when starting from D, like "D-E"
t = ___   # True or False: the MST contains the shortest path between every pair of vertices

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 4 and q == 8 and r == [\"A-B\", \"C-D\"], \"p, q and r\"\n    assert s == \"D-E\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: which edges matter?

When weights tie, a graph can have several MSTs. Classify every edge:

- **critical**: in **every** MST (removing it makes the MST total go up, or disconnects the graph);
- **sometimes**: in some MSTs but not all (forcing it into the tree first still gives the minimum total);
- **never**: in no MST.

Write `classify(vertices, edges)`: a dict from each edge's **index** in `edges` to `"critical"`, `"sometimes"` or `"never"`, by running Kruskal's algorithm with an edge **left out** (does the total rise, or the graph disconnect?) and with an edge **forced in** first (is the total still the minimum?). That's O(E × E log E), fine for a few hundred edges. Then `all_msts_total(vertices, edges)`: the MST total (raise `ValueError` if disconnected). The checks compare with a brute force over every spanning tree on small graphs.

In [ ]:
def all_msts_total(vertices, edges):
    ___

def classify(vertices, edges):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef brute(vs, es):\n    trees = []\n    for combo in itertools.combinations(range(len(es)), len(vs) - 1):\n        parent = {v: v for v in vs}\n        def find(x):\n            while parent[x] != x:\n                x = parent[x]\n            return x\n        ok = True\n        for i in combo:\n            u, v, w = es[i]\n            if find(u) == find(v):\n                ok = False\n                break\n            parent[find(u)] = find(v)\n        if ok:\n            trees.append((sum(es[i][2] for i in combo), set(combo)))\n    best = min(t for t, _ in trees)\n    msts = [s for t, s in trees if t == best]\n    out = {}\n    for i in range(len(es)):\n        inside = sum(1 for s in msts if i in s)\n        out[i] = \"critical\" if inside == len(msts) else \"sometimes\" if inside else \"never\"\n    return best, out\ndef test_classify():\n    \"classifying\"\n    edges = [(\"A\", \"B\", 1), (\"B\", \"C\", 1), (\"C\", \"D\", 2), (\"A\", \"D\", 2), (\"A\", \"C\", 3), (\"D\", \"E\", 1)]\n    assert classify(\"ABCDE\", edges) == {0: \"critical\", 1: \"critical\", 2: \"sometimes\", 3: \"sometimes\", 4: \"never\", 5: \"critical\"}\n    assert all_msts_total(\"ABCDE\", edges) == 5\n    rng = random.Random(5)\n    for _ in range(80):\n        vs = list(range(rng.randint(2, 5)))\n        es = [(a, b, rng.randint(1, 3)) for a in vs for b in vs if a < b and rng.random() < 0.8]\n        parent = {v: v for v in vs}\n        def find(x):\n            while parent[x] != x:\n                x = parent[x]\n            return x\n        for u, v, w in es:\n            parent[find(u)] = find(v)\n        if len({find(v) for v in vs}) > 1:\n            continue\n        total, want = brute(vs, es)\n        assert all_msts_total(vs, es) == total and classify(vs, es) == want, (vs, es)")

## Warm-up

Cheapest first: A-C (1), B-D (2), B-C (3). The next, A-B (4), would close a loop (A, B and C are already joined), so it's skipped. Three edges join four places for a cost of 6, and no other choice is cheaper.

## Spanning trees

For a connected undirected graph with V vertices:

- A **spanning tree** is a set of edges that connects every vertex and contains no cycle. It always has exactly **V − 1** edges (Week 50 Day 1: a tree with V vertices).
- A **minimum spanning tree** is a spanning tree with the smallest total weight. There can be several with the same total; if all edge weights are **different**, the MST is **unique**.
- If the graph isn't connected, there's no spanning tree; the same algorithms then give a **minimum spanning forest**, one tree per component.

Don't confuse an MST with shortest paths: the MST minimises the **total** of all its edges, not the distance between any particular pair. The path between two vertices in the MST can be much longer than their shortest path.

```text
   A ---1--- B           MST: A-B, B-C (total 2)
    \       /            but the shortest path from A to C is the direct edge, 1.5,
    1.5    1             not A-B-C, which is 2.
      \   /
        C
```

## Kruskal's algorithm

Joseph Kruskal's algorithm (1956):

1. Sort the edges by weight.
2. Go through them, cheapest first, keeping each edge whose ends are in **different** components of what's been kept so far (so it can't close a cycle), and skipping the rest.
3. Stop after V − 1 edges are kept.

"Different components?" and "join them" are exactly union-find's `find` and `union`. Sorting costs O(E log E); the union-find work is effectively O(E). So the total is **O(E log E)**, which is the same as O(E log V).

Putting the weight first in each tuple makes `sorted` order the edges by weight with no key function needed.

## Prim's algorithm

Prim's algorithm (Robert Prim, 1957; found earlier by Vojtěch Jarník in 1930) grows **one** tree:

1. Start the tree with any vertex.
2. Repeatedly add the **cheapest edge** that joins a vertex in the tree to one outside it.

It's Dijkstra's algorithm with one change: the priority of an outside vertex is the weight of the cheapest edge reaching it from the tree, not its distance from the start. With `heapq` and lazy deletion, it's **O(E log V)**.

For **dense** graphs (every pair connected, like points on a map where any two can be joined), E is about V², and a simpler version without a heap is better: keep, for each outside vertex, its cheapest edge to the tree in a plain dict, and each step scan for the smallest. That's **O(V²)**, with no V² edges ever built.

Same total, 30, as Kruskal's algorithm. The trees may differ when weights tie (here two edges weigh 7 and two weigh 8), but every MST has the same total.

## Why greedy works

Greedy choices often go wrong (Week 14), so why are they right here? Because of the **cut property**: split the vertices into any two groups (a **cut**); the cheapest edge crossing between the groups belongs to some MST. (If an MST didn't use it, adding it would close a cycle that crosses the cut somewhere else, by an edge at least as heavy; swapping the two gives a spanning tree no heavier.)

- Prim's algorithm applies it to the cut "tree so far" against "everything else".
- Kruskal's algorithm applies it to the cut around one of the two components an edge joins: when it's considered, it's the cheapest edge left, so the cheapest crossing that cut.

A matching **cycle property** says the heaviest edge on any cycle (if it's strictly the heaviest) is in no MST, which is why Kruskal's algorithm can skip edges that close cycles.

## Uses

- **Network design**: the cheapest cabling, piping or roads connecting everything (the original problem: Otakar Borůvka designed an electricity network for Moravia in 1926 with the first MST algorithm).
- **Maximum spanning tree**: sort the other way (or negate weights) to keep the **strongest** links.
- **Bottleneck routes**: in a maximum spanning tree, the path between two vertices maximises the weakest edge (Monday's widest path); in a minimum spanning tree, it minimises the heaviest edge.
- **Clustering**: run Kruskal's algorithm but stop when k components remain (equivalently, delete the k − 1 heaviest MST edges). Points close together end up in the same cluster, and the gap between clusters (the **spacing**) is as large as possible. This is **single-linkage clustering**.

## Practice

## Quiz

## Daily challenge

## Before you go

Mark six places you know on a sketch map, guess the distances between them, and find the cheapest way to connect them all by hand with Kruskal's algorithm. Tomorrow: grids as graphs, going further than last week, with A* search and searches over richer states.